# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [1]:
%idle_timeout 2880
%glue_version 4.0
%worker_type G.1X
%number_of_workers 2

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.7 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 4.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 2
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 2
Idle Timeout: 2880
Session ID: ccc01669-4f40-4a82-bff6-adfd30596efd
Applying the following default arguments:
--glue_kernel_version 1.0.7
--enable-glue-datacatalog true
Waiting for session ccc01669-4f40-4a82-bff6-adfd30596efd to get into ready status...
Session ccc01669-4f40-4a82-bff6-adfd30596efd ha

In [2]:
dyf = glueContext.create_dynamic_frame_from_catalog(
    database='lds_raw',
    table_name='asignacion_tarifa'
)

df = dyf.toDF()
df.show(10)
df.printSchema()

+--------------------+-------------+---------+
|id_asignacion_tarifa|id_suministro|id_tarifa|
+--------------------+-------------+---------+
|                   1|      1000000|       16|
|                   2|      1000001|       22|
|                   3|      1000002|       12|
|                   4|      1000003|       27|
|                   5|      1000004|       13|
|                   6|      1000005|       22|
|                   7|      1000006|        8|
|                   8|      1000007|       17|
|                   9|      1000008|       23|
|                  10|      1000009|        4|
+--------------------+-------------+---------+
only showing top 10 rows

root
 |-- id_asignacion_tarifa: long (nullable = true)
 |-- id_suministro: long (nullable = true)
 |-- id_tarifa: long (nullable = true)

/opt/amazon/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py:127: UserWarning: DataFrame constructor is internal. Do not directly use it.


In [3]:
total = df.count()
print("Total filas:", total)


Total filas: 12160


In [4]:
import pyspark.sql.functions as F

print("=== NULOS POR COLUMNA ===")
df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()


=== NULOS POR COLUMNA ===
+--------------------+-------------+---------+
|id_asignacion_tarifa|id_suministro|id_tarifa|
+--------------------+-------------+---------+
|                   0|            0|        0|
+--------------------+-------------+---------+


In [8]:
string_cols = [c for c, t in df.dtypes if t == "string"]

print("=== BLANCOS POR COLUMNA ===")
df.select([
    F.count(F.when(F.col(c) == "", c)).alias(c)
    for c in string_cols
]).show()


=== BLANCOS POR COLUMNA ===
++
||
++
||
||
||
||
||
||
||
||
||
||
||
||
||
||
||
||
||
||
||
||
++
only showing top 20 rows


In [12]:
df.toPandas().isnull().sum().sum()

0
